# Replicate ChatGPT Price Forecasting

This notebook gives a concise, reader-friendly tour of the project workflow and the core outputs used in the replication.

## Summary
The Lopez-Lira and Tang (2023) paper "Can ChatGPT Forecast Stock Price Movements? Return Predictability and Large Language Models" documents the capability of LLMs, such as ChatGPT, to predict stock market reactions from news headlines without direct financial training. The papers results suggest forecasting ability generally increases with model size and strategy returns decline as LLM usage within the finanical domain increases. We aim to replicate their results, specifically looking at strategy hit rates, and portfolio returns results.

## Methodology at a glance
- Data foundation: pull CRSP prices and RavenPack headlines from WRDS.
- Data cleaning: map entities to tickers, remove low-quality/duplicate signals, and align headline timing to trading logic.
- NLP labeling: submit batched headlines to OpenAI and parse labels into structured sentiment scores.
- Portfolio construction: convert firm-day sentiment into long/short return series under multiple sample restrictions.
- Reporting: generate summary tables and figures used in the write-up.

## Pipeline context for this notebook
The data shown below comes from the `doit` pipeline in `dodo.py`.
- `pull:crsp_stock` runs `pull_CRSP_stock.py` and writes `CRSP_stock_daily.parquet` and `CRSP_unique_tickers.parquet`.
- `pull:ravenpack` runs `pull_ravenpack.py` and writes `RAVENPACK.parquet`.
- `clean_data:clean_ravenpack` runs `clean_ravenpack.py` and writes `RAVENPACK_cleaned.parquet`.
- `process:generate_batched_requests`, `process:submit_headlines_to_openai`, and `process:create_firm_day_score` produce `daily_headline_polarity.parquet`.
- Downstream scripts `create_portfolios.py` and `create_table1.py` generate return panels and final table CSVs used in this guide.

## Step 0. Setup and Key Paths
Start by confirming which pipeline artifacts exist before loading data. This prevents downstream errors and makes the notebook reproducible across machines and collaborators.

The code imports project settings from `settings.py`, builds the canonical file map, and prints an existence table so we can verify which stages of the pipeline have already been run.

In [6]:
from pathlib import Path
import pandas as pd
import json

from notebook_helper import generate_single_request_jsonl, get_rp_timing_stats
from settings import config

DATA_DIR = Path(config("DATA_DIR"))
OUTPUT_DIR = Path(config("OUTPUT_DIR"))

paths = {
    "CRSP Stock Data": DATA_DIR / "CRSP_stock_daily.parquet",
    "RavenPack Full": DATA_DIR / "RAVENPACK.parquet",
    "RavenPack Clean": DATA_DIR / "RAVENPACK_cleaned.parquet",
    "Daily Headline Scores": DATA_DIR / "daily_headline_polarity.parquet",
    "Portfolio Returns": DATA_DIR / "portfolio_daily_returns.parquet",
    "Table1 (Oct 2021 - May 2024)": DATA_DIR / "table1_overnight_paper_sample.csv",
    "Table1 (Oct 2021 - March 2026)": DATA_DIR / "table1_overnight_full_sample.csv",
}

pd.DataFrame({
    "file": list(paths.keys()),
    "exists": [p.exists() for p in paths.values()],
})

,file,exists
0,CRSP Stock Data,True
1,RavenPack Full,True
2,RavenPack Clean,True
3,Daily Headline Scores,True
4,Portfolio Returns,True
5,Table1 (Oct 2021 - May 2024),True
6,Table1 (Oct 2021 - March 2026),True


## Step 1. Load Cleaned RavenPack and CRSP

In order to replicate the results in this paper, we need market data from CRSP and news data from RavenPack. We are ignoring intraday data. Additionally, we use the RPA Entity Mapping File (wrds_rpa_company_mappings) in order to facilitate the mapping of RavenPack entities to company tickers.

We pulled data between 2021-10-1 and 2026-03-01 for this project. Our replication is concerned with the paper's timebounds of 2021-10-01 to 2024-05-31, and additional analysis was completed on the full sample.

- The <b>CRSP dataset</b> contains stock market data, including prices, returns, and shares outstanding, for securities traded on major U.S. exchanges. This dataset is essential for measuring market performance and calculating variables such as market equity.
- The <b>RavenPack dataset</b>, using over 40,000 sources, provides real-time news analytics, including sentiment analysis and event data focused on business and financial applications. Data includes news and social media content, allowing for comprehensive analysis of financial markets.
- The <b>RPA Entity Mapping File</b> provides a variety of security identifiers (ex. ISINs, CUSIPs, etc.) to allow you to identify the 90,000+ companies.

**Pipeline step that generated this data**
- `CRSP_stock_daily.parquet` is generated by `pull:crsp_stock` via `pull_CRSP_stock.py`.
- `RAVENPACK_cleaned.parquet` is generated by `clean_data:clean_ravenpack` via `clean_ravenpack.py` (after `pull:ravenpack`).

Our pulled CRSP dataset is limited to the world of stocks specified in the paper:

```SQL
WHERE 
    ( 
        primaryexch IN ( 'N', 'A', 'Q' ) AND
        conditionaltype = 'RW' AND
        tradingstatusflg = 'A' AND
        dlycaldt >= '{start_date}' AND
        dlycaldt <= '{end_date}'
        {permno_filter}
    ) ;
```

The `clean_ravenpack` process was designed to replicate the data cleaning procedure described in the paper. This includes filtering RavenPack data to match CRSP tickers, ensure a baseline relevancy rating of 0.60, deduplicating firm-day headlines using Optimal String Alignment (OSA) similarity, and aligning headline timestamps to Eastern Time. Intraday headlines are excluded, and overnight headlines are adjusted to ensure proper alignment with trading day per teh overnight cutoff at 4:00PM ET. This step ensures that the cleaned dataset is consistent with the methodology outlined in the paper.

In [2]:
rp = pd.read_parquet(paths["RavenPack Clean"]) if paths["RavenPack Clean"].exists() else pd.DataFrame()
crsp = pd.read_parquet(paths["CRSP Stock Data"]) if paths["CRSP Stock Data"].exists() else pd.DataFrame()

print("RavenPack cleaned shape:", rp.shape)
print("CRSP Stock Data shape:", crsp.shape)

display(rp.head(3))
display(crsp.head(3))
del crsp

RavenPack cleaned shape: (209714, 8)
CRSP Stock Data shape: (7467730, 8)


,rp_entity_id,map_ticker,entity_name,timestamp_utc,rpa_date_utc,timestamp_et,headline_date,headline
0,00067A,HUM,Humana Inc.,2025-01-09 21:30:00.623,2025-01-09,2025-01-09 16:30:00.623000-05:00,2025-01-10,Humana Inc. to Release Fourth Quarter 2024 Res...
1,00067A,HUM,Humana Inc.,2025-11-12 13:45:00.460,2025-11-12,2025-11-12 08:45:00.460000-05:00,2025-11-12,ProgenyHealth Announces Collaboration with Hum...
2,00067A,HUM,Humana Inc.,2025-07-31 11:00:04.767,2025-07-31,2025-07-31 07:00:04.767000-04:00,2025-07-31,Exact Sciences and Humana Expand Colorectal Ca...


,permno,permco,ticker,primaryexch,date,dlycap,dlyopen,dlyclose
0,10026,7976,JJSF,Q,2021-10-01,2932065.76,153.85,153.64
1,10028,7978,ELA,A,2021-10-01,109584.75,4.11,4.07
2,10032,7980,PLXS,Q,2021-10-01,2554520.76,89.59,91.08


### Headline Timing Diagnostics

In [3]:
stats, table = get_rp_timing_stats(rp)

for key, t in [("RavenPack Stats", stats), ("RavenPack Timing Table", table)]:
    if not t.empty:
        print(key + ":")
        display(t)

Intraday rows remaining (expected ~0): 0
RavenPack Stats:


n_rows                                         209714
n_tickers                                        5505
min_ts               2021-09-30 20:21:21.593000-04:00
max_ts               2026-02-28 18:00:03.460000-05:00
min_headline_date                          2021-10-01
max_headline_date                          2026-03-01
dtype: object

RavenPack Timing Table:


,sample,rows total,headline dates rolled over
0,Oct 2021 - May 2024,130111,23798
1,Oct 2021 - Feb 2026,209714,38707


## Step 3. Generate Batched Requests (Single-Row Example)

**Pipeline step that generated this data**
- `openai_headline_requests.*.jsonl` files are generated by `process:generate_batched_requests` via `generate_batched_requests.py`.
- `id_to_row_mapping.*.json` files are generated by `process:generate_batched_requests` via `generate_batched_requests.py`.

This step demonstrates the `generate_batched_requests.py` logic on a single RavenPack row to replicate the pipeline behavior.

In order to retrieve OpenAI responses to our headline prompts, we must create batch files for asynchornous submission. The RavenPack data is broken up into chunks (40,000 being the default) and each request is a single line of json (referred to as JSONL). We also create a separate mapping file for ease of processing later on.

In [18]:
jsonl_content, mapping_content = generate_single_request_jsonl(rp.head(1).copy())

Wrote requests jsonl: /tmp/tmpv1h5nt42/openai_headline_requests.single.jsonl
Number of headlines batched: 1
Wrote id to row mapping json: /tmp/tmpv1h5nt42/id_to_row_mapping.single.json


In [30]:
print("JSONL content inline:")
display(jsonl_content)

JSONL content inline:


'{"custom_id": "rp-0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "gpt-3.5-turbo", "temperature": 0, "messages": [{"role": "system", "content": "Forget all your previous instructions. Pretend you are a financial expert. You are a financial expert with stock recommendation experience. Answer \\"YES\\" if good news, \\"NO\\" if bad news, or \\"UNKNOWN\\" if uncertain in the first line. Then elaborate with one short and concise sentence on the next line."}, {"role": "user", "content": "Is this headline good or bad for the stock price of Humana Inc. in the short term?\\nHeadline: Humana Inc. to Release Fourth Quarter 2024 Results on February 11, 2025"}]}}\n'

In [32]:
print("JSONL content formatted:")
print(json.dumps(json.loads(jsonl_content), sort_keys=True, indent=2, separators=(",", ": ")))

JSONL content formatted:
{
  "body": {
    "messages": [
      {
        "content": "Forget all your previous instructions. Pretend you are a financial expert. You are a financial expert with stock recommendation experience. Answer \"YES\" if good news, \"NO\" if bad news, or \"UNKNOWN\" if uncertain in the first line. Then elaborate with one short and concise sentence on the next line.",
        "role": "system"
      },
      {
        "content": "Is this headline good or bad for the stock price of Humana Inc. in the short term?\nHeadline: Humana Inc. to Release Fourth Quarter 2024 Results on February 11, 2025",
        "role": "user"
      }
    ],
    "model": "gpt-3.5-turbo",
    "temperature": 0
  },
  "custom_id": "rp-0",
  "method": "POST",
  "url": "/v1/chat/completions"
}


In [ ]:
print("ID to Row Mapping:")
print(json.dumps(json.loads(mapping_content), sort_keys=True, indent=2, separators=(",", ": ")))

ID to Row Mapping:
{
  "rp-0": {
    "date": "2025-01-10",
    "entity_name": "Humana Inc.",
    "ticker": "HUM"
  }
}


## Step 4. Submit Headlines to OpenAI Batch API

**Pipeline step that generated this data**
- `openai_headline_batch_output.*.jsonl` files are generated by `process:submit_headlines_to_openai` via `submit_headlines_to_openai.py`.
- `openai_headline_batch_metadata.*.json` files are generated by `process:submit_headlines_to_openai` via `submit_headlines_to_openai.py`.

All batch request files matching `openai_headline_requests.*.jsonl` are then submitted to OpenAI via the `process:submit_headlines_to_openai` step, with an SLA of 24 hours for a response. The pipeline blocks while waiting for a completion status for all submitted files. Results are written to the `DATA_DIR` as `openai_headline_batch_output.*.jsonl` with corresponding metadata files `openai_headline_batch_metadata.*.json`.

Below is a success response from OpenAI.

In [43]:
data = pd.read_json(DATA_DIR / "openai_headline_batch_output.1.jsonl", lines=True).head(1)
display(data.iloc[0]['response'])

{'status_code': 200,
 'request_id': '20c04669-5262-44e6-8fca-cc48e5b5d47b',
 'body': {'id': 'chatcmpl-DJUCVBfr6XgXN8m415PXlAhTb9NaR',
  'object': 'chat.completion',
  'created': 1773536695,
  'model': 'gpt-3.5-turbo-0125',
  'choices': [{'index': 0,
    'message': {'role': 'assistant',
     'content': 'UNKNOWN\nThe stock price may fluctuate depending on the content of the results and market expectations.',
     'refusal': None,
     'annotations': []},
    'logprobs': None,
    'finish_reason': 'stop'}],
  'usage': {'prompt_tokens': 117,
   'completion_tokens': 19,
   'total_tokens': 136,
   'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
   'completion_tokens_details': {'reasoning_tokens': 0,
    'audio_tokens': 0,
    'accepted_prediction_tokens': 0,
    'rejected_prediction_tokens': 0}},
  'service_tier': 'default',
  'system_fingerprint': None}}

# TODO: complete below steps

## Step 5. Firm-Day Sentiment Output

**Methodology**
Convert model-level headline classifications into a daily ticker panel that can be merged to CRSP and traded in portfolio tests.

**Pipeline step that generated this data**
`daily_headline_polarity.parquet` is generated by `process:create_firm_day_score` in `create_firmday_score.py`, using OpenAI batch outputs created by:
- `process:generate_batched_requests` (`generate_batched_requests.py`)
- `process:submit_headlines_to_openai` (`submit_headlines_to_openai.py`)

**Code discussion**
The code loads the firm-day score file, previews rows, and reports the distribution of `score` values (`-1`, `0`, `1`) to check label balance before portfolio formation.

In [47]:
scores = pd.read_parquet(paths["Daily Headline Scores"]) if paths["Daily Headline Scores"].exists() else pd.DataFrame()
print("Scores shape:", scores.shape)
display(scores.head(5))

if not scores.empty:
    summary = scores["score"].value_counts(dropna=False).rename_axis("score").reset_index(name="count")
    summary["share"] = (summary["count"] / summary["count"].sum()).round(4)
    summary.sort_values("score",inplace=True, ignore_index=True)
    summary = summary.rename(index={0: 'NO', 1: 'UNKNOWN', 2: 'YES'})
    display(summary)

Scores shape: (176968, 5)


,ticker,date,n_headlines,score_sum,score
0,A,2021-11-18,1,1,1
1,A,2021-11-23,2,1,1
2,A,2022-01-27,1,0,0
3,A,2022-02-16,1,1,1
4,A,2022-02-23,3,2,1


,score,count,share
NO,-1,9296,0.0525
UNKNOWN,0,118923,0.6720
YES,1,48749,0.2755


## Step 5. Portfolio Returns and Table Outputs

**Methodology**
Map sentiment signals into implementable long/short return series and summarize results in publication-ready tables.

**Pipeline step that generated this data**
- `portfolio_daily_returns.parquet` is generated by `create_portfolios.py`.
- `table1_overnight_paper_sample.csv` and `table1_overnight_full_sample.csv` are generated by `create_table1.py` from the return panel and score inputs.

**Code discussion**
The code previews the daily portfolio return panel, then loads and previews both Table 1 variants so the reader can connect intermediate return construction to final reported statistics.

In [50]:
port = pd.read_parquet(paths["Portfolio Returns"]) if paths["Portfolio Returns"].exists() else pd.DataFrame()
print("Portfolio returns:", port.shape)
display(port.head(5))

for key in ["Table1 (Oct 2021 - May 2024)", "Table1 (Oct 2021 - March 2026)"]:
    p = paths[key]
    if p.exists():
        t = pd.read_csv(p)
        print("\n" + key + ":", t.shape if p.exists() else "not found")
        print("Summary:")
        print("  > Trading Days: ", t.iloc[3]["Trading Days"] if p.exists() else "")
        print("  > Firm-Day Observations: ", int(t.iloc[3]["Firm-Day Observations"]) if p.exists() else "")
        display(t.head(3).drop(columns=["Firm-Day Observations"], errors="ignore"))
    else:
        print(key + ": not found")

Portfolio returns: (1087, 20)


,date,n_neg,n_neu,n_pos,n_total,ret_long,n_long,n_short,ret_short,ret_ir_long,ret_ir_short,ret_ls_restricted,ret_ls_not_small,ret_ls_price_gt_5,ret_mkt_vw,trade_long,trade_short,trade_ls,ret_ls,ret_ir_ls
0,2021-10-01,3,37,19,59,0.0,0,0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,True,True,True,0.000000,0.000000
1,2021-10-04,3,83,37,123,-0.001595,32,2,-0.001542,0.003042,0.051936,-0.004477,-0.004432,0.000915,-0.010693,True,True,True,-0.003137,0.054978
2,2021-10-05,5,107,33,145,0.020661,29,5,0.026466,0.006686,0.00104,0.066331,0.066341,0.059321,0.006008,True,True,True,0.047127,0.007726
3,2021-10-06,3,113,45,161,0.00936,40,2,0.033815,0.008709,0.023499,-0.013591,-0.013591,-0.015007,0.011786,True,True,True,0.043175,0.032208
4,2021-10-07,3,93,43,139,0.001959,41,1,-0.054348,0.007832,0.114533,-0.063873,-0.062724,-0.059204,0.002413,True,True,True,-0.052389,0.122366



Table1 (Oct 2021 - May 2024): (4, 8)
Summary:
  > Trading Days:  670
  > Firm-Day Observations:  112092


,Portfolio,Initial Reaction Hit Rate (%),Initial Reaction Mean Return (%),Drift Hit Rate (%),Drift Mean Return (%),Drift Sharpe Ratio,Trading Days
0,Long-Short Portfolio,92.836,3.300,52.239,0.240,1.458,670
1,Long-Only Portfolio,84.179,1.973,48.507,0.020,0.195,670
2,Short-Only Portfolio,85.629,1.353,54.491,0.215,1.214,668



Table1 (Oct 2021 - March 2026): (4, 8)
Summary:
  > Trading Days:  1087
  > Firm-Day Observations:  176968


,Portfolio,Initial Reaction Hit Rate (%),Initial Reaction Mean Return (%),Drift Hit Rate (%),Drift Mean Return (%),Drift Sharpe Ratio,Trading Days
0,Long-Short Portfolio,93.100,4.381,51.702,0.165,0.912,1087
1,Long-Only Portfolio,85.465,3.004,47.470,-0.036,-0.321,1087
2,Short-Only Portfolio,84.885,1.392,55.945,0.187,0.993,1085


## Step 6. Code Tour (Where each step lives)

**Methodology**
The pipeline is intentionally modular: each script owns one transformation and writes a named artifact, which makes the workflow testable and easy to audit.

**Pipeline mapping from script to generated artifacts**
- `pull_CRSP_stock.py` -> `CRSP_stock_daily.parquet`, `CRSP_unique_tickers.parquet`
- `pull_ravenpack.py` -> `RAVENPACK.parquet`
- `clean_ravenpack.py` -> `RAVENPACK_cleaned.parquet`
- `generate_batched_requests.py` -> `openai_headline_requests.*.jsonl`, `id_to_row_mapping.*.json`
- `submit_headlines_to_openai.py` -> `openai_headline_batch_output.*.jsonl` and metadata files
- `create_firmday_score.py` -> `daily_headline_polarity.parquet`
- `create_portfolios.py` -> `portfolio_daily_returns.parquet`
- `create_table1.py` -> `table1_overnight_paper_sample.csv`, `table1_overnight_full_sample.csv`

**Code discussion**
Use this mapping as a guide for debugging or extending the project: if a table looks wrong, you can trace it directly to the upstream script and file that produced it.